# 02 — Odd/even lap validation of GCaMP place-cell maps

This notebook validates spatial tuning by splitting laps into odd and even laps. It is designed to run after the main Suite2p/GCaMP preprocessing workflow.

The analysis:

1. loads cleaned transient traces and behaviour-aligned variables;
2. detects lap boundaries from position resets;
3. computes odd-lap and even-lap ratemaps;
4. correlates odd/even maps for each cell;
5. saves validation metrics and QC figures.

## 1. Imports and repository paths

As in the main notebook, this section adds the repository `functions/` folder to the Python path so that `helper_functions.py` can be imported after cloning the repository.

In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() in {"notebooks", "main"} else NOTEBOOK_DIR

FUNCTIONS_DIR = REPO_ROOT / "functions"
if FUNCTIONS_DIR.exists():
    sys.path.insert(0, str(FUNCTIONS_DIR))

from helper_functions import bin_data_trace

## 2. User settings

These settings should match the output folders created by the main notebook. The odd/even notebook loads the processed outputs rather than re-running the full place-cell shuffle test.

In [ ]:
BASE_DIR = Path(r"C:\path\to\VR Analysis\Cohort 7\data")
DATE = "250408"
MICE = ["DG31", "DG32", "DG33", "DG35", "DG36"]

# Sessions to validate.
SESSIONS = ("pre", "post")

# These should match the output folder names from the main notebook.
OUT_PRE  = "output_data_pre_fix"
OUT_POST = "output_data_post_fix"

# Spatial map settings.
TRACK_START_CM = 0
TRACK_LENGTH_CM = 295
N_POSITION_BINS = 59
VELOCITY_THRESHOLD_CM_S = 5
GAUSSIAN_SIGMA_BINS = 2
PLACE_CELL_ALPHA = 0.05

# Lap detection threshold.
# A lap reset is detected when position drops by more than this amount.
LAP_DROP_THRESHOLD_CM = 50

## 3. Loading processed outputs

The main notebook saves cleaned transients (`Fc3_cleaned.csv`), position, velocity, timestamps, p-values, ratemaps, and cell IDs for each mouse/session. This section provides small loading functions for those outputs.

In [ ]:
def mouse_dir(mouse: str) -> Path:
    return BASE_DIR / DATE / mouse


def output_dir(mouse: str, session: str) -> Path:
    if session == "pre":
        return mouse_dir(mouse) / OUT_PRE
    if session == "post":
        return mouse_dir(mouse) / OUT_POST
    raise ValueError("session must be 'pre' or 'post'")


def load_processed_session(mouse: str, session: str):
    """Load the processed arrays saved by the main notebook."""
    save_path = output_dir(mouse, session)

    Fc3_df = pd.read_csv(save_path / "Fc3_cleaned.csv")
    Fc3_cleaned = np.asarray(Fc3_df.T)
    cell_ids = np.asarray(Fc3_df.columns)

    coords = np.loadtxt(save_path / "coords_trim.csv", delimiter=",")
    velocity = np.loadtxt(save_path / "velocity_trim.csv", delimiter=",")
    trace_time = np.loadtxt(save_path / "trace_time_trim.csv", delimiter=",")

    p_values = np.loadtxt(save_path / "p_values.csv", delimiter=",")
    ratemaps_norm = np.loadtxt(save_path / "ratemaps_norm.csv", delimiter=",")

    return {
        "save_path": save_path,
        "Fc3_cleaned": Fc3_cleaned,
        "cell_ids": cell_ids,
        "coords": coords,
        "velocity": velocity,
        "trace_time": trace_time,
        "p_values": p_values,
        "ratemaps_norm": ratemaps_norm,
    }

## 4. Lap detection and odd/even splitting

Lap boundaries are detected from sharp drops in the animal's position trace. Laps are then separated into odd and even sets based on their order in the session.

In [ ]:
def find_lap_slices(coords, lap_drop_threshold_cm=LAP_DROP_THRESHOLD_CM):
    """Return a list of slice objects, one per detected lap."""
    coords = np.asarray(coords)
    dpos = np.diff(coords)

    lap_starts = np.where(dpos <= -abs(lap_drop_threshold_cm))[0] + 1
    lap_starts = np.unique(np.r_[0, lap_starts])
    lap_ends = np.r_[lap_starts[1:], coords.shape[0]]

    lap_slices = [
        slice(int(start), int(end))
        for start, end in zip(lap_starts, lap_ends)
        if end > start
    ]

    return lap_slices


def combine_slices(slices, n_frames):
    """Convert a list of slices into a boolean frame mask."""
    mask = np.zeros(n_frames, dtype=bool)
    for sl in slices:
        mask[sl] = True
    return mask


def get_odd_even_masks(coords):
    """Return odd-lap and even-lap frame masks."""
    lap_slices = find_lap_slices(coords)

    odd_laps = lap_slices[0::2]
    even_laps = lap_slices[1::2]

    odd_mask = combine_slices(odd_laps, len(coords))
    even_mask = combine_slices(even_laps, len(coords))

    return odd_mask, even_mask, lap_slices

## 5. Compute odd/even ratemaps

For each cell, activity is binned by position separately for odd and even laps. The same running-speed threshold and smoothing parameters used in the main notebook are applied here.

In [ ]:
def compute_ratemaps_for_mask(Fc3_cleaned, coords, velocity, trace_time, frame_mask):
    """Compute normalised ratemaps for a subset of frames."""

    bin_edges = np.linspace(TRACK_START_CM, TRACK_LENGTH_CM, N_POSITION_BINS + 1)

    subset_mask = frame_mask & (velocity > VELOCITY_THRESHOLD_CM_S)

    coords_active = coords[subset_mask]
    time_per_frame = float(trace_time.max() / trace_time.shape[0])

    frames_per_bin, _ = np.histogram(coords_active, bins=bin_edges)
    time_per_bin = frames_per_bin * time_per_frame

    smooth_time_binned = gaussian_filter(
        time_per_bin.astype(float),
        sigma=GAUSSIAN_SIGMA_BINS,
        truncate=2,
    ).reshape(-1, 1)
    smooth_time_binned_safe = np.maximum(smooth_time_binned, 1e-12)

    n_cells = Fc3_cleaned.shape[0]
    ratemaps = np.zeros((n_cells, N_POSITION_BINS))
    ratemaps_norm = np.zeros((n_cells, N_POSITION_BINS))

    for i in range(n_cells):
        trace_active = Fc3_cleaned[i, subset_mask]

        if np.sum(trace_active) == 0 or coords_active.size == 0:
            continue

        binned_mean_dff = bin_data_trace(coords_active, trace_active, bin_edges)
        smooth_binned_mean_dff = gaussian_filter(
            binned_mean_dff.astype(float),
            sigma=GAUSSIAN_SIGMA_BINS,
            truncate=2,
        )

        binned_ratemap = smooth_binned_mean_dff / smooth_time_binned_safe

        ratemaps[i, :] = binned_ratemap.ravel()

        mx = float(np.max(binned_ratemap))
        ratemaps_norm[i, :] = (binned_ratemap.ravel() / mx) if mx > 0 else 0

    return ratemaps, ratemaps_norm


def correlate_odd_even_maps(odd_maps, even_maps):
    """Compute Pearson correlation between odd- and even-lap ratemaps for each cell."""
    n_cells = odd_maps.shape[0]
    correlations = np.full(n_cells, np.nan)

    for i in range(n_cells):
        odd = odd_maps[i, :]
        even = even_maps[i, :]

        if np.all(odd == 0) or np.all(even == 0):
            continue

        if np.std(odd) == 0 or np.std(even) == 0:
            continue

        correlations[i] = pearsonr(odd, even)[0]

    return correlations

## 6. Plotting functions

These plots provide quick QC outputs for the odd/even validation: a distribution of odd/even correlations and heatmaps sorted by each cell's peak in the odd-lap map.

In [ ]:
def plot_odd_even_correlation_distribution(correlations, save_path, mouse, session):
    """Save a histogram of odd/even ratemap correlations."""
    fig, ax = plt.subplots(figsize=(4, 3))

    valid = correlations[np.isfinite(correlations)]
    ax.hist(valid, bins=25)
    ax.axvline(np.nanmedian(correlations), linestyle="--", linewidth=1)

    ax.set_title(f"{mouse} {session}: odd/even map correlations")
    ax.set_xlabel("Pearson r")
    ax.set_ylabel("Cells")

    plt.tight_layout()
    outpath = Path(save_path) / f"{mouse}_{session}_odd_even_correlation_distribution.png"
    plt.savefig(outpath, dpi=300)
    plt.close(fig)
    print(f"[FIG] Saved {outpath}")


def plot_odd_even_heatmaps(odd_maps, even_maps, save_path, mouse, session, cell_mask=None):
    """Save odd/even heatmaps sorted by odd-lap peak position."""

    if cell_mask is None:
        cell_mask = np.ones(odd_maps.shape[0], dtype=bool)

    odd = odd_maps[cell_mask]
    even = even_maps[cell_mask]

    if odd.size == 0:
        print(f"[FIG] No cells to plot for {mouse} {session}")
        return

    peak_bins = np.argmax(odd, axis=1)
    sort_idx = np.argsort(peak_bins)

    odd_sorted = odd[sort_idx]
    even_sorted = even[sort_idx]

    fig, axes = plt.subplots(1, 2, figsize=(6, 6), sharey=True)

    sns.heatmap(odd_sorted, ax=axes[0], cmap="viridis", vmin=0, vmax=1, cbar=False)
    sns.heatmap(even_sorted, ax=axes[1], cmap="viridis", vmin=0, vmax=1, cbar=True)

    axes[0].set_title("Odd laps")
    axes[1].set_title("Even laps")

    for ax in axes:
        ax.set_xlabel("Track position (bins)")
        ax.set_ylabel("Cells")
        ax.tick_params(length=0)

    plt.tight_layout()
    outpath = Path(save_path) / f"{mouse}_{session}_odd_even_ratemaps.png"
    plt.savefig(outpath, dpi=300)
    plt.close(fig)
    print(f"[FIG] Saved {outpath}")

## 7. Run odd/even validation

This section loops through all selected mice and sessions, computes odd/even ratemaps, saves the correlations, and produces QC figures. By default, heatmaps are plotted for significant place cells from the main notebook.

In [ ]:
def run_odd_even_validation(mice, sessions=SESSIONS):
    """Run odd/even ratemap validation for all mice and sessions."""

    for mouse in mice:
        for session in sessions:
            try:
                data = load_processed_session(mouse, session)

                save_path = data["save_path"]
                Fc3_cleaned = data["Fc3_cleaned"]
                coords = data["coords"]
                velocity = data["velocity"]
                trace_time = data["trace_time"]
                p_values = data["p_values"]
                cell_ids = data["cell_ids"]

                odd_mask, even_mask, lap_slices = get_odd_even_masks(coords)

                print(f"[RUN] {mouse} {session}: {len(lap_slices)} laps detected")

                odd_maps, odd_maps_norm = compute_ratemaps_for_mask(
                    Fc3_cleaned, coords, velocity, trace_time, odd_mask
                )
                even_maps, even_maps_norm = compute_ratemaps_for_mask(
                    Fc3_cleaned, coords, velocity, trace_time, even_mask
                )

                correlations = correlate_odd_even_maps(odd_maps_norm, even_maps_norm)

                # Save outputs.
                np.savetxt(save_path / "odd_lap_ratemaps_norm.csv", odd_maps_norm, delimiter=",")
                np.savetxt(save_path / "even_lap_ratemaps_norm.csv", even_maps_norm, delimiter=",")
                np.savetxt(save_path / "odd_even_map_correlations.csv", correlations, delimiter=",")

                validation_df = pd.DataFrame({
                    "cell_id": cell_ids,
                    "place_cell_p": p_values,
                    "is_place_cell": p_values <= PLACE_CELL_ALPHA,
                    "odd_even_r": correlations,
                })
                validation_df.to_csv(save_path / "odd_even_validation_summary.csv", index=False)

                # Plot all significant place cells only.
                pc_mask = (p_values <= PLACE_CELL_ALPHA) & np.isfinite(correlations)

                plot_odd_even_correlation_distribution(correlations[pc_mask], save_path, mouse, session)
                plot_odd_even_heatmaps(odd_maps_norm, even_maps_norm, save_path, mouse, session, cell_mask=pc_mask)

                print(f"[DONE] {mouse} {session} -> {save_path}")

            except Exception as e:
                print(f"[ERROR] {mouse} {session}: {e}")

## 8. Run validation

In [ ]:
run_odd_even_validation(MICE)
